# Data Workflow & Acquisition
# Hospital Appointment No-Show Prediction

## 0. Setup

In [49]:
import os
import sys
import platform
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)

Python: 3.13.7
Platform: Linux-6.17.0-14-generic-x86_64-with-glibc2.42
pandas: 3.0.1


## 1. The DS lifecycle (workflow map)

**Problem Statement:** 
Analyzing appointment records in order to predict missed medical appointments.

**Success Metric:** F1-score for predicting where "no_show = yes".

## 2. Project structure (repeatable and collaborative)

In [51]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
CONFIGS = PROJECT_ROOT / "configs"

for p in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, REPORTS, CONFIGS]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

PosixPath('/media/irseora/external/Repos/Data-Science/Week-1')

### Add a simple README

In [52]:
readme_text = """# Demo Project: Data Workflow & Acquisition

## Purpose
Teaching example for:
- DS lifecycle
- Project structure
- Reproducibility basics
- Reading CSV/Excel reliably

## How to run
1. Create environment (optional): `python -m venv .venv` then install requirements
2. Run `Assignment1_DataWorkflow.ipynb` cells in order

## Data folders
- `data/raw/`: immutable source files
- `data/interim/`: intermediate outputs
- `data/processed/`: analysis-ready outputs

## Outputs
- `reports/`: tables/figures for sharing
"""

(PROJECT_ROOT / "README.md").write_text(readme_text)
print("Wrote:", PROJECT_ROOT / "README.md")

Wrote: /media/irseora/external/Repos/Data-Science/Week-1/README.md


## 3. Reproducibility essentials

In [53]:
# Freeze a minimal requirements file
requirements = [
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    "openpyxl"  # needed for reading/writing .xlsx with pandas
]
(PROJECT_ROOT / "requirements.txt").write_text("\n".join(requirements) + "\n")
print((PROJECT_ROOT / "requirements.txt").read_text())

pandas==3.0.1
numpy==2.4.2
openpyxl



In [54]:
# Control randomness
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Demonstrate determinism
np.random.rand(3)

array([0.37454012, 0.95071431, 0.73199394])

In [55]:
# Configuration file
config = {
    "seed": SEED,
    "raw_csv": "appointments_raw.csv",
    "date_format": "%Y-%m-%d",
    "currency_columns": ["fee"],
    "validation_rules": {
        "age_min": 0,
        "age_max": 120,
        "fee_min": 0,
        "waiting_days_min": 0
    }
}

config_path = CONFIGS / "config.json"
config_path.write_text(json.dumps(config, indent=2))

print("Wrote:", config_path)
print(config_path.read_text())

Wrote: /media/irseora/external/Repos/Data-Science/Week-1/configs/config.json
{
  "seed": 42,
  "raw_csv": "appointments_raw.csv",
  "date_format": "%Y-%m-%d",
  "currency_columns": [
    "fee"
  ],
  "validation_rules": {
    "age_min": 0,
    "age_max": 120,
    "fee_min": 0,
    "waiting_days_min": 0
  }
}


## 4. Create demo data (CSV)

In [56]:
# Generate a small "sales" dataset with intentional issues
import re
from datetime import datetime, timedelta

def messy_date(dt, rng):
    formats = [
        "%Y-%m-%d",
        "%b %d %Y",
        "%d-%b-%Y",
        "%Y/%m/%d",
    ]

    if rng.random() < 0.02:
        return "not_a_date"
    return dt.strftime(rng.choice(formats))

def messy_fee(val, rng):
    symbols = [ "$", "€", "£", "" ]
    symbol = rng.choice(symbols)

    # Decimals
    if rng.random() < 0.5:
        amount = f"{val:.2f}"
    else:
        amount = f"{int(val)}"

    # Thousands separator
    if val > 999 and rng.random() < 0.3:
        amount = f"{val:,.2f}"

    return f"{symbol}{amount}"

def messy_waiting_days(days, rng):
    r = rng.random()
    if r < 0.70:
        return str(days)
    if r < 0.85:
        return f"{days} days"
    if r < 0.92:
        return ""
    if r < 0.97:
        return "N/A"
    return str(float(days))

def generate_data(
        raw_dir=DATA_RAW, 
        filename="appointments_raw.csv", 
        n_rows=1000, seed=SEED
):
    raw_dir = Path(raw_dir)
    rng = random.Random(seed)
    np.random.seed(seed)

    start = pd.Timestamp("2025-01-01")
    end = pd.Timestamp("2025-12-31")
    total_days = (end - start).days

    appointment_types = ["Checkup", "Follow-up", "Blood Test", "Specialist", "Procedure"]
    clinics = ["Central", "North", "East", "West"]
    channels = ["Phone", "Online", "Walk-in"]
    base_ids = [f"APT-{100000+i}" for i in range(n_rows)]
    booking_dates = []
    appointment_dates = []
    waiting_days_int = []

    for _ in range(n_rows):
        book = start + pd.Timedelta(days=rng.randint(0, total_days))
        wait = max(0, int(np.random.normal(10, 7)))
        appt = book + pd.Timedelta(days=wait)

        booking_dates.append(book)
        appointment_dates.append(appt)
        waiting_days_int.append(wait)

    ages = np.random.normal(42, 18, n_rows).round().astype(int)
    ages = np.clip(ages, -10, 200)

    for idx in np.random.choice(n_rows, size=max(1, n_rows // 25), replace=False):
        ages[idx] = -9999

    fee_vals = np.random.lognormal(3.6, 0.5, n_rows)
    fee_vals = (fee_vals / np.percentile(fee_vals, 95)) * 250
    fee_vals = np.clip(fee_vals, 10, 500)
    neg_indices = np.random.choice(n_rows, size=max(1, n_rows // 100), replace=False)
    fee_vals[neg_indices] *= -1
    miss_fee_indices = np.random.choice(n_rows, size=max(1, n_rows // 30), replace=False)

    appt_type = [rng.choice(appointment_types) for _ in range(n_rows)]

    wait_arr = np.array(waiting_days_int)
    base_prob = 0.12 + (wait_arr / 60) * 0.25
    type_bump = np.array([0.05 if t in ["Specialist", "Procedure"] else 0 for t in appt_type])
    p_no_show = np.clip(base_prob + type_bump, 0.05, 0.60)

    no_show = np.where(np.random.rand(n_rows) < p_no_show, "Yes", "No")

    booking_date_str = [messy_date(d, rng) for d in booking_dates]
    appointment_date_str = [messy_date(d, rng) for d in appointment_dates]
    waiting_days_messy = [messy_waiting_days(d, rng) for d in waiting_days_int]

    fee_messy = []
    for i, v in enumerate(fee_vals):
        if i in set(miss_fee_indices):
            fee_messy.append("" if rng.random() < 0.7 else "N/A")
        else:
            fee_messy.append(messy_fee(float(v), rng))

    age_messy = []
    for a in ages:
        if a == -9999:
            age_messy.append("" if rng.random() < 0.7 else "unknown")
        else:
            s = str(a)
            if rng.random() < 0.2:
                s = " " + s + " "
            age_messy.append(s)

    df = pd.DataFrame({
        "appointment_id": base_ids,
        "booking_date": booking_date_str,
        "appointment_date": appointment_date_str,
        "age": age_messy,
        "waiting_days": waiting_days_messy,
        "fee": fee_messy,
        "appointment_type": appt_type,
        "clinic": [rng.choice(clinics) for _ in range(n_rows)],
        "booking_channel": [rng.choice(channels) for _ in range(n_rows)],
        "no_show": no_show
    })

    dup_count = max(1, n_rows // 40)
    dup_rows = df.sample(n=dup_count, random_state=seed)
    df = pd.concat([df, dup_rows], ignore_index=True)

    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    out_path = raw_dir / filename
    if out_path.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = raw_dir / f"{out_path.stem}_{stamp}{out_path.suffix}"

    df.to_csv(out_path, index=False)
    return out_path

raw_data_path = generate_data()

## 5. Data acquisition from CSV
### 5.1 Basic read

In [57]:
df_raw = pd.read_csv(raw_data_path)
print(df_raw.shape)
df_raw.head()

(1025, 10)


,appointment_id,booking_date,appointment_date,age,waiting_days,fee,appointment_type,clinic,booking_channel,no_show
0,APT-100527,09-Aug-2025,2025-08-19,70,10,$63.52,Procedure,Central,Walk-in,No
1,APT-100359,2025-12-10,Dec 14 2025,45,4,£80,Procedure,Central,Walk-in,No
2,APT-100447,03-Jul-2025,Jul 06 2025,53,3,$77.55,Blood Test,Central,Online,No
3,APT-100031,2025/01/04,Jan 26 2025,12,22,£250.00,Follow-up,North,Phone,Yes
4,APT-100621,10-Mar-2025,23-Mar-2025,unknown,13 days,€80,Follow-up,East,Phone,No


### 5.2 Inspect schema and quality quickly

In [58]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   appointment_id    1025 non-null   str  
 1   booking_date      1025 non-null   str  
 2   appointment_date  1025 non-null   str  
 3   age               989 non-null    str  
 4   waiting_days      922 non-null    str  
 5   fee               990 non-null    str  
 6   appointment_type  1025 non-null   str  
 7   clinic            1025 non-null   str  
 8   booking_channel   1025 non-null   str  
 9   no_show           1025 non-null   str  
dtypes: str(10)
memory usage: 80.2 KB


In [59]:
df_raw.isna().sum().sort_values(ascending=False)

waiting_days        103
age                  36
fee                  35
appointment_id        0
booking_date          0
appointment_date      0
appointment_type      0
clinic                0
booking_channel       0
no_show               0
dtype: int64

In [60]:
df_raw.duplicated().sum()

np.int64(25)

### 5.3 Robust Parsing

In [61]:
def clean_mydata(df_raw):
    df = df_raw.copy()

    df = df.rename(columns={"age": "patient_age"})

    # Dates
    df["booking_date"] = pd.to_datetime(df["booking_date"], errors="coerce", format="mixed", dayfirst=True)
    df["appointment_date"] = pd.to_datetime(df["appointment_date"], errors="coerce", format="mixed", dayfirst=True)

    # Currencies
    df["fee"] = (
        df["fee"].astype(str)
            .str.replace(r"[$€£]", "", regex=True)
            .str.replace(",", "", regex=False)
            .str.strip()
    )
    df["fee"] = pd.to_numeric(df["fee"], errors="coerce")

    # Numerics
    df["patient_age"] = (
        df["patient_age"].astype(str).str.strip()
    )
    df["patient_age"]= pd.to_numeric(df["patient_age"], errors="coerce").astype("Int64")

    df["waiting_days"] = (
        df["waiting_days"].astype(str)
        .str.extract(r"(\d+)", expand=False)  # pulls "12" from "12 days"
    )
    df["waiting_days"] = pd.to_numeric(df["waiting_days"], errors="coerce").astype("Int64")

    # Derived column
    df["is_minor"] = df["patient_age"] < 18

    # Sort and reset index
    df = df.sort_values(["appointment_date", "appointment_id"]).reset_index(drop=True)

    return df

df = clean_mydata(df_raw)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   appointment_id    1025 non-null   str           
 1   booking_date      1001 non-null   datetime64[us]
 2   appointment_date  1004 non-null   datetime64[us]
 3   patient_age       982 non-null    Int64         
 4   waiting_days      922 non-null    Int64         
 5   fee               990 non-null    float64       
 6   appointment_type  1025 non-null   str           
 7   clinic            1025 non-null   str           
 8   booking_channel   1025 non-null   str           
 9   no_show           1025 non-null   str           
 10  is_minor          982 non-null    boolean       
dtypes: Int64(2), boolean(1), datetime64[us](2), float64(1), str(5)
memory usage: 84.2 KB


### 5.4. Validation Checks

In [62]:
df_valid = df.copy()

# R1: Appointmen id must not be missing
missing_ids = df_valid["appointment_id"].isna().sum()
assert missing_ids == 0, f"Found {missing_ids} missing appointment_id values."

# R2: Checking for not_a_date + drop
bad_appt_dates = df_valid["appointment_date"].isna().sum()
bad_booking_dates = df_valid["booking_date"].isna().sum()
print("Unparsed appointment_date:", bad_appt_dates)
print("Unparsed booking_date:", bad_booking_dates)
df_valid = df_valid.dropna(subset=["appointment_date"]).copy()

# R3: Checking for negative fee
neg_fee = (df_valid["fee"] < 0).sum()
print("Negative fee rows:", neg_fee)
df_valid.loc[df_valid["fee"] < 0, "fee"] = np.nan

# R4: Realistic age
out_of_range_age = ((df_valid["patient_age"] < 0) | (df_valid["patient_age"] > 120)).sum()
print("Out-of-range age rows:", out_of_range_age)
df_valid.loc[(df_valid["patient_age"] < 0) | (df_valid["patient_age"] > 120), "patient_age"] = pd.NA

# Rule 5: Duplicate appointment ids
dup_count = df_valid.duplicated(subset=["appointment_id"]).sum()
print("Duplicate appointment_id rows:", dup_count)
df_valid = df_valid.drop_duplicates(subset=["appointment_id"], keep="first").copy()

# Recompute derived column
df_valid["is_minor"] = df_valid["patient_age"] < 18

# Final assertions
assert df_valid.duplicated(subset=["appointment_id"]).sum() == 0, "Duplicates still present!"
assert (df_valid["fee"].dropna() >= 0).all(), "Still found negative fees!"
assert df_valid["appointment_date"].isna().sum() == 0, "appointment_date still has NaT after dropping!"

Unparsed appointment_date: 21
Unparsed booking_date: 24
Negative fee rows: 9
Out-of-range age rows: 11
Duplicate appointment_id rows: 25


### 5.5. Saving

In [63]:
name = "appointments"
interim_path = DATA_INTERIM / f"{name}_interim.csv"
processed_path = DATA_PROCESSED / f"{name}_processed.csv"

df.to_csv(interim_path, index=False)
df_valid.to_csv(processed_path, index=False)

#### Summary

In [64]:
summary = pd.DataFrame([
    {"metric": "rows_raw_loaded", "value": int(len(df_raw))},
    {"metric": "rows_interim", "value": int(len(df))},
    {"metric": "rows_processed", "value": int(len(df_valid))},

    {"metric": "unparsed_appointment_date_interim", "value": int(df["appointment_date"].isna().sum())},
    {"metric": "unparsed_booking_date_interim", "value": int(df["booking_date"].isna().sum())},

    {"metric": "duplicate_appointment_id_interim", "value": int(df.duplicated(subset=["appointment_id"]).sum())},
    {"metric": "duplicate_appointment_id_processed", "value": int(df_valid.duplicated(subset=["appointment_id"]).sum())},

    {"metric": "missing_fee_processed", "value": int(df_valid["fee"].isna().sum())},
    {"metric": "missing_patient_age_processed", "value": int(df_valid["patient_age"].isna().sum())},

    {"metric": "no_show_rate_yes_processed", "value": float((df_valid["no_show"] == "Yes").mean())},
    {"metric": "minor_rate_processed", "value": float(df_valid["is_minor"].mean())},
])

summary_path = REPORTS / "summary_table.csv"
summary.to_csv(summary_path, index=False)
summary

,metric,value
0,rows_raw_loaded,1025.000000
1,rows_interim,1025.000000
2,rows_processed,979.000000
3,unparsed_appointment_date_interim,21.000000
4,unparsed_booking_date_interim,24.000000
5,duplicate_appointment_id_interim,25.000000
6,duplicate_appointment_id_processed,0.000000
7,missing_fee_processed,41.000000
8,missing_patient_age_processed,50.000000
9,no_show_rate_yes_processed,0.181818
